# DNABERT-2 Heavy Training on Google Colab

This notebook prepares a larger ClinVar GRCh38 alternate-sequence dataset and fine-tunes DNABERT-2 on a Colab GPU.

Research/education only. This is not a medical diagnosis tool.

## 1. GPU Check

In [ ]:
import subprocess
import torch

print("nvidia-smi output:")
subprocess.run(["nvidia-smi"], check=False)

cuda_available = torch.cuda.is_available()
print("torch CUDA available:", cuda_available)

if not cuda_available:
    raise RuntimeError("No GPU is available. In Colab, go to Runtime > Change runtime type > GPU, then rerun.")

print("GPU name:", torch.cuda.get_device_name(0))

## 2. Clone GitHub Repo

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "PASTE_YOUR_GITHUB_REPO_URL_HERE"

if REPO_URL == "PASTE_YOUR_GITHUB_REPO_URL_HERE":
    raise ValueError("Paste your GitHub repository URL into REPO_URL before running this cell.")

repo_name = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
repo_path = Path("/content") / repo_name

if not repo_path.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo_path)], check=True)
else:
    print(f"Repository already exists: {repo_path}")

os.chdir(repo_path)
print("Project root:", Path.cwd())

## 3. Install Dependencies

In [ ]:
!python -m pip install --upgrade pip
!pip install -q torch transformers datasets accelerate scikit-learn pandas numpy safetensors tqdm biopython requests joblib

## 4. Optional Google Drive Mount

In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = True
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/variant-risk-explainer")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print("Drive output folder:", DRIVE_OUTPUT_DIR)
else:
    print("Google Drive mount skipped.")

## 5. Prepare Larger ClinVar Dataset

In [ ]:
TARGET_TOTAL = 10000
!python training/prepare_larger_clinvar_dataset.py --target_total {TARGET_TOTAL}

Optional: only run this 20k cell after 10k preparation and training work correctly.

In [ ]:
RUN_20K = False

if RUN_20K:
    !python training/prepare_larger_clinvar_dataset.py --target_total 20000
else:
    print("20k preparation skipped. Set RUN_20K = True when ready.")

## 6. Check Dataset

In [ ]:
!python training/check_large_dataset.py

## 7. Audit Alternate Sequence Encoding

Expected conclusion: `The sequence appears to contain alternate alleles.`

In [ ]:
!python training/audit_variant_sequence_encoding.py

## 8. GPU Training

In [ ]:
%%bash
python training/train_local_dnabert2.py \
  --sample_size 0 \
  --epochs 5 \
  --batch_size 4 \
  --grad_accum_steps 4 \
  --learning_rate 2e-5 \
  --freeze_encoder true \
  --unfreeze_last_n_layers 4 \
  --use_class_weights true \
  --center_crop true \
  --tune_threshold true \
  --eval_accumulation_steps 8 \
  --save_eval_each_epoch false \
  --eval_subset_size 0

## 9. Full Evaluation

In [ ]:
!python training/evaluate_saved_model.py --tune_threshold true

## 10. Save Outputs to Google Drive

In [ ]:
from pathlib import Path
import shutil

source_dir = Path("training/outputs/dnabert2_clinvar")
drive_output = Path("/content/drive/MyDrive/variant-risk-explainer")

if not drive_output.exists():
    print("Google Drive output folder is not available. Run the Drive mount cell first.")
else:
    drive_output.mkdir(parents=True, exist_ok=True)
    items_to_copy = [
        source_dir / "final_model",
        source_dir / "metrics.json",
        source_dir / "full_eval_metrics.json",
        source_dir / "final_dnabert2_clinvar_model.zip",
    ]

    for item in items_to_copy:
        if not item.exists():
            print(f"Skipping missing item: {item}")
            continue

        destination = drive_output / item.name
        if item.is_dir():
            if destination.exists():
                shutil.rmtree(destination)
            shutil.copytree(item, destination)
        else:
            shutil.copy2(item, destination)
        print(f"Copied {item} -> {destination}")

## 11. Download Model Artifacts

In [ ]:
from pathlib import Path
from google.colab import files

output_dir = Path("training/outputs/dnabert2_clinvar")
zip_path = output_dir / "final_dnabert2_clinvar_model.zip"

if not zip_path.exists():
    !cd training/outputs/dnabert2_clinvar && zip -r final_dnabert2_clinvar_model.zip final_model metrics.json full_eval_metrics.json

files.download(str(zip_path))